# Data bn tanishuv

In [78]:
import pandas as pd 
df=pd.read_csv('fifa_player_performance_market_value.csv')

In [79]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2800 entries, 0 to 2799
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player_id                 2800 non-null   int64  
 1   player_name               2800 non-null   str    
 2   age                       2800 non-null   int64  
 3   nationality               2800 non-null   str    
 4   club                      2800 non-null   str    
 5   position                  2800 non-null   str    
 6   overall_rating            2800 non-null   int64  
 7   potential_rating          2800 non-null   int64  
 8   matches_played            2800 non-null   int64  
 9   goals                     2800 non-null   int64  
 10  assists                   2800 non-null   int64  
 11  minutes_played            2800 non-null   int64  
 12  market_value_million_eur  2800 non-null   float64
 13  contract_years_left       2800 non-null   int64  
 14  injury_prone       

In [80]:
df.isnull().sum()

player_id                   0
player_name                 0
age                         0
nationality                 0
club                        0
position                    0
overall_rating              0
potential_rating            0
matches_played              0
goals                       0
assists                     0
minutes_played              0
market_value_million_eur    0
contract_years_left         0
injury_prone                0
transfer_risk_level         0
dtype: int64

In [81]:
df['TOTAL_CONTRIBUTION'] = df['goals'] + df['assists']
df['POTENTIAL_GROWTH'] = df['potential_rating'] - df['overall_rating']
df['VALUE_PER_RATING'] = df['market_value_million_eur'] / (df['overall_rating'] + 0.1)
df['IS_VETERAN'] = df['age'].apply(lambda x: 1 if x > 30 else 0)
df['SHORT_CONTRACT'] = df['contract_years_left'].apply(lambda x: 1 if x <= 1 else 0)

Encoding,Scaling va Cleaner 

In [82]:
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
class DataPreprocessing:
    def __init__(self,df):
        self.df=df
    def tozala(self):
        for col in self.df.columns:
            if self.df[col].isnull().any():
                if self.df[col].dtype=='str':
                    self.df[col].fillna(self.df[col].mode()[0],inplace=True)                
                else:
                    self.df[col].fillna(self.df[col].mean(),inplace=True)
        return self
    def encodla(self):
        encoder=LabelEncoder()
        for col in self.df.columns:
            if self.df[col].dtype=='str':
                if self.df[col].nunique()<=5:
                    dummies=pd.get_dummies(self.df[col],prefix=col,dtype=int)
                    self.df=pd.concat([self.df.drop(columns=col),dummies],axis=1)
                else:
                    self.df[col]=encoder.fit_transform(self.df[col])
        return self
    def scale_qil(self):
        scaler=MinMaxScaler()
        for col in self.df.columns:
            num_col=self.df.select_dtypes(include=['int64','float64']).columns
            self.df[num_col]=scaler.fit_transform(self.df[num_col])
        return self
    

In [83]:
dp=DataPreprocessing(df)
dp.tozala().encodla().scale_qil()
preprocessed_data=dp.df

In [84]:
preprocessed_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2800 entries, 0 to 2799
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   player_id                   2800 non-null   float64
 1   player_name                 2800 non-null   float64
 2   age                         2800 non-null   float64
 3   nationality                 2800 non-null   float64
 4   club                        2800 non-null   float64
 5   position                    2800 non-null   float64
 6   overall_rating              2800 non-null   float64
 7   potential_rating            2800 non-null   float64
 8   matches_played              2800 non-null   float64
 9   goals                       2800 non-null   float64
 10  assists                     2800 non-null   float64
 11  minutes_played              2800 non-null   float64
 12  market_value_million_eur    2800 non-null   float64
 13  contract_years_left         2800 non-null   

# Train
# Prediction
# Model Selection
# Linear R
# Decision T
# Random F
# SVM 

In [85]:
X = df.drop(['market_value_million_eur'], axis=1)
y = df['market_value_million_eur']

In [86]:
import pandas as pd
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error

df['injury_prone'] = df['injury_prone'].map({'No': 0, 'Yes': 1})
df['transfer_risk_level'] = df['transfer_risk_level'].map({'Low': 0, 'Medium': 1, 'High': 2})

X = df.select_dtypes(include=['int64', 'float64']).drop(['market_value_million_eur', 'player_id'], axis=1, errors='ignore')
y = df['market_value_million_eur']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "SVM (SVR)": SVR(kernel='rbf')
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    
    score = model.score(X_test, y_test)
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    
    results.append([name, f"{score:.3f}", f"{mae:.3f}", f"{mse:.3f}"])

headers = ["Model", "R2 Score", "MAE", "MSE"]
print(tabulate(results, headers=headers, tablefmt="grid"))

+-------------------+------------+--------+----------+
| Model             |   R2 Score |    MAE |      MSE |
+===================+============+========+==========+
| Linear Regression |      0.983 |  5.423 |   49.471 |
+-------------------+------------+--------+----------+
| Decision Tree     |      0.997 |  2.187 |    8.643 |
+-------------------+------------+--------+----------+
| Random Forest     |      0.999 |  0.999 |    1.939 |
+-------------------+------------+--------+----------+
| SVM (SVR)         |     -0.003 | 46.899 | 2866.85  |
+-------------------+------------+--------+----------+
